# Benchmarking: Metropolis-Hastings vs Hamiltonian Monte Carlo

## Introduction

Hamiltonian Monte Carlo (HMC) is a state-of-the-art MCMC method that uses gradient information to propose efficient moves. This notebook compares:

1. **Metropolis-Hastings (MH)** - Our custom implementation
2. **Hamiltonian Monte Carlo (HMC)** - Using the [Mici](https://github.com/matt-graham/mici) library

We'll compare them on:
- Sampling efficiency (Effective Sample Size)
- Mode exploration
- Autocorrelation
- Computational cost

## Setup

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
import time
from src.metropolis_hastings import MetropolisHastings
from src.target_distributions import (
    mixture_of_gaussians_log_pdf, 
    banana_shaped_log_pdf,
    get_distribution
)
from src.utils import (
    compute_effective_sample_size, 
    plot_contour, 
    plot_autocorrelation,
    create_comparison_figure
)
import seaborn as sns
sns.set_style('whitegrid')

%matplotlib inline

## 1. Target Distributions

We'll test on two challenging distributions:
- **Mixture of Gaussians**: Multi-modal
- **Banana-shaped**: Non-linear geometry

In [ ]:
# Visualize both target distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

distributions = {
    'Mixture of Gaussians': mixture_of_gaussians_log_pdf,
    'Banana-shaped': banana_shaped_log_pdf
}

for ax, (name, log_pdf) in zip(axes, distributions.items()):
    x = np.linspace(-6, 6, 100)
    y = np.linspace(-6, 6, 100)
    X, Y = np.meshgrid(x, y)
    grid_points = np.column_stack([X.ravel(), Y.ravel()])
    
    log_vals = np.array([log_pdf(p) for p in grid_points])
    vals = np.exp(log_vals - np.max(log_vals))
    pdf_grid = vals.reshape(X.shape)
    
    ax.contourf(X, Y, pdf_grid, levels=30, cmap='viridis')
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 2. Setting up HMC with Mici

Mici requires defining the negative potential energy (negative log density). Let's set up the HMC sampler.

In [ ]:
import mici
from mici.systems import EuclideanMetricSystem
from mici.integrators import LeapfrogIntegrator
from mici.samplers import StaticHamiltonianMonteCarlo

def create_hmc_sampler(target_log_pdf, n_dim=2, step_size=0.1, n_steps=20):
    """
    Create an HMC sampler using Mici.
    
    Parameters
    ----------
    target_log_pdf : callable
        Log probability density function
    n_dim : int
        Dimensionality
    step_size : float
        Leapfrog step size
    n_steps : int
        Number of leapfrog steps per proposal
    
    Returns
    -------
    sampler : mici.samplers.StaticHamiltonianMonteCarlo
        Configured HMC sampler
    system : mici.systems.EuclideanMetricSystem
        The system (needed for initial state)
    """
    def neg_potential(x):
        """Negative potential energy (negative log density)."""
        return -target_log_pdf(x)
    
    system = EuclideanMetricSystem(neg_potential, ndim=n_dim)
    integrator = LeapfrogIntegrator(system, step_size=step_size)
    sampler = StaticHamiltonianMonteCarlo(integrator, n_steps=n_steps)
    
    return sampler, system


def run_hmc(sampler, system, n_samples, burn_in=0, initial_state=None):
    """
    Run HMC sampler.
    
    Returns
    -------
    samples : np.ndarray
        Generated samples
    acceptance_rate : float
        Acceptance rate
    """
    if initial_state is None:
        state = system.initial_state
    else:
        state = system.initial_state
        state.pos = initial_state.copy()
    
    all_samples = []
    n_accepted = 0
    
    # Burn-in
    for _ in range(burn_in):
        new_state = sampler.sample(state)
        if new_state.n_accepted:
            n_accepted += 1
        state = new_state
        all_samples.append(state.pos.copy())
    
    # Collect samples
    samples = []
    for _ in range(n_samples):
        new_state = sampler.sample(state)
        if new_state.n_accepted:
            n_accepted += 1
        state = new_state
        samples.append(state.pos.copy())
    
    total_proposals = burn_in + n_samples
    acceptance_rate = n_accepted / total_proposals
    
    return np.array(samples), acceptance_rate

## 3. Comparison on Mixture of Gaussians

### 3.1 Run Both Samplers

In [ ]:
n_samples = 5000
burn_in = 1000

# Metropolis-Hastings
print("Running Metropolis-Hastings...")
mh_sampler = MetropolisHastings(
    target_log_pdf=mixture_of_gaussians_log_pdf,
    proposal_scale=0.5,
    n_dim=2,
    adapt_scale=True
)

start = time.time()
mh_samples, mh_diag = mh_sampler.sample(
    n_samples=n_samples,
    burn_in=burn_in,
    thin=1,
    progress_bar=True
)
mh_time = time.time() - start

# HMC
print("\nRunning HMC...")
hmc_sampler, hmc_system = create_hmc_sampler(
    mixture_of_gaussians_log_pdf,
    n_dim=2,
    step_size=0.1,
    n_steps=20
)

start = time.time()
hmc_samples, hmc_acceptance = run_hmc(
    hmc_sampler, 
    hmc_system, 
    n_samples=n_samples, 
    burn_in=burn_in
)
hmc_time = time.time() - start

# Compute ESS
mh_ess = compute_effective_sample_size(mh_samples)
hmc_ess = compute_effective_sample_size(hmc_samples)

# Results
print("\n" + "="*50)
print("Comparison on Mixture of Gaussians:")
print("="*50)
print(f"MH  - Acceptance: {mh_diag['acceptance_rate']:.3f}, ESS: {mh_ess:.0f}, Time: {mh_time:.2f}s")
print(f"HMC - Acceptance: {hmc_acceptance:.3f}, ESS: {hmc_ess:.0f}, Time: {hmc_time:.2f}s")
print(f"ESS improvement: {hmc_ess/mh_ess:.2f}x")
print(f"Efficiency (ESS/second): MH={mh_ess/mh_time:.0f}, HMC={hmc_ess/hmc_time:.0f}")

### 3.2 Visual Comparison

In [ ]:
fig = create_comparison_figure(
    mh_samples,
    hmc_samples,
    mixture_of_gaussians_log_pdf,
    mh_ess=mh_ess,
    hmc_ess=hmc_ess,
    save_path='../results/figures/mh_vs_hmc_mixture.png'
)
plt.show()

### 3.3 Autocorrelation Comparison

In [ ]:
from src.utils import compute_autocorrelation

max_lag = 50
mh_autocorr = compute_autocorrelation(mh_samples, max_lag)
hmc_autocorr = compute_autocorrelation(hmc_samples, max_lag)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for dim, ax in enumerate(axes):
    ax.plot(mh_autocorr[:, dim], label='MH', alpha=0.7)
    ax.plot(hmc_autocorr[:, dim], label='HMC', alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.set_title(f'Dimension {dim+1} Autocorrelation')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/mh_vs_hmc_autocorrelation_mixture.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Comparison on Banana-Shaped Distribution

The banana distribution is particularly challenging for MH with isotropic proposals.

In [ ]:
n_samples = 5000
burn_in = 1000

# Metropolis-Hastings
print("Running Metropolis-Hastings on Banana...")
mh_sampler_banana = MetropolisHastings(
    target_log_pdf=banana_shaped_log_pdf,
    proposal_scale=0.3,
    n_dim=2,
    adapt_scale=True
)

start = time.time()
mh_samples_banana, mh_diag_banana = mh_sampler_banana.sample(
    n_samples=n_samples,
    burn_in=burn_in,
    thin=1,
    progress_bar=True
)
mh_time_banana = time.time() - start

# HMC on Banana
print("\nRunning HMC on Banana...")
hmc_sampler_banana, hmc_system_banana = create_hmc_sampler(
    banana_shaped_log_pdf,
    n_dim=2,
    step_size=0.08,  # Smaller step for banana
    n_steps=20
)

start = time.time()
hmc_samples_banana, hmc_acceptance_banana = run_hmc(
    hmc_sampler_banana, 
    hmc_system_banana, 
    n_samples=n_samples, 
    burn_in=burn_in
)
hmc_time_banana = time.time() - start

# Compute ESS
mh_ess_banana = compute_effective_sample_size(mh_samples_banana)
hmc_ess_banana = compute_effective_sample_size(hmc_samples_banana)

# Results
print("\n" + "="*50)
print("Comparison on Banana-Shaped Distribution:")
print("="*50)
print(f"MH  - Acceptance: {mh_diag_banana['acceptance_rate']:.3f}, ESS: {mh_ess_banana:.0f}, Time: {mh_time_banana:.2f}s")
print(f"HMC - Acceptance: {hmc_acceptance_banana:.3f}, ESS: {hmc_ess_banana:.0f}, Time: {hmc_time_banana:.2f}s")
print(f"ESS improvement: {hmc_ess_banana/mh_ess_banana:.2f}x")
print(f"Efficiency (ESS/second): MH={mh_ess_banana/mh_time_banana:.0f}, HMC={hmc_ess_banana/hmc_time_banana:.0f}")

### 4.1 Visual Comparison on Banana

In [ ]:
fig = create_comparison_figure(
    mh_samples_banana,
    hmc_samples_banana,
    banana_shaped_log_pdf,
    mh_ess=mh_ess_banana,
    hmc_ess=hmc_ess_banana,
    save_path='../results/figures/mh_vs_hmc_banana.png'
)
plt.show()

### 4.2 Autocorrelation Comparison on Banana

In [ ]:
mh_autocorr_banana = compute_autocorrelation(mh_samples_banana, max_lag)
hmc_autocorr_banana = compute_autocorrelation(hmc_samples_banana, max_lag)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for dim, ax in enumerate(axes):
    ax.plot(mh_autocorr_banana[:, dim], label='MH', alpha=0.7)
    ax.plot(hmc_autocorr_banana[:, dim], label='HMC', alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.set_title(f'Dimension {dim+1} Autocorrelation (Banana)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/mh_vs_hmc_autocorrelation_banana.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Summary Comparison Table

Let's compile all results into a single comparison table.

In [ ]:
import pandas as pd

# Create comparison table
comparison_data = {
    'Distribution': ['Mixture of Gaussians', 'Mixture of Gaussians', 
                     'Banana-shaped', 'Banana-shaped'],
    'Sampler': ['MH', 'HMC', 'MH', 'HMC'],
    'Acceptance Rate': [mh_diag['acceptance_rate'], hmc_acceptance,
                        mh_diag_banana['acceptance_rate'], hmc_acceptance_banana],
    'ESS': [mh_ess, hmc_ess, mh_ess_banana, hmc_ess_banana],
    'Time (s)': [mh_time, hmc_time, mh_time_banana, hmc_time_banana],
    'ESS/Time': [mh_ess/mh_time, hmc_ess/hmc_time, 
                 mh_ess_banana/mh_time_banana, hmc_ess_banana/hmc_time_banana]
}

df = pd.DataFrame(comparison_data)
print("\nPerformance Comparison Summary:")
print("="*70)
print(df.to_string(index=False))
print("="*70)

### 5.1 Bar Chart Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ESS comparison
x = np.arange(len(df))
width = 0.35

ess_values = df[df['Sampler'] == 'MH']['ESS'].values
hmc_ess_values = df[df['Sampler'] == 'HMC']['ESS'].values
distributions = ['Mixture', 'Banana']

ax1.bar(x - width/2, ess_values, width, label='MH', alpha=0.7)
ax1.bar(x + width/2, hmc_ess_values, width, label='HMC', alpha=0.7)
ax1.set_xlabel('Distribution')
ax1.set_ylabel('Effective Sample Size')
ax1.set_title('ESS Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(distributions)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Efficiency comparison
mh_efficiency = df[df['Sampler'] == 'MH']['ESS/Time'].values
hmc_efficiency = df[df['Sampler'] == 'HMC']['ESS/Time'].values

ax2.bar(x - width/2, mh_efficiency, width, label='MH', alpha=0.7)
ax2.bar(x + width/2, hmc_efficiency, width, label='HMC', alpha=0.7)
ax2.set_xlabel('Distribution')
ax2.set_ylabel('ESS / Second')
ax2.set_title('Efficiency Comparison (ESS/Second)')
ax2.set_xticks(x)
ax2.set_xticklabels(distributions)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/mh_vs_hmc_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Conclusion

**Key Findings:**

1. **HMC is significantly more efficient**:
   - Higher ESS for the same number of samples
   - Lower autocorrelation
   - Better exploration of challenging geometries

2. **MH struggles with**:
   - Multi-modal distributions (mode switching)
   - Non-linear geometries (banana shape)
   - Requires careful tuning of proposal scale

3. **HMC advantages**:
   - Uses gradient information for smart proposals
   - Can take longer steps with high acceptance
   - Better for complex, correlated distributions

4. **Trade-offs**:
   - HMC requires gradient computation (more complex)
   - HMC has more tuning parameters (step size, number of steps)
   - MH is simpler and more generally applicable

**When to use each**:
- **Use MH**: Simple distributions, when gradients are unavailable, or for quick exploration
- **Use HMC**: Complex distributions, high-dimensional problems, when efficiency matters

**Next Steps**:
- Try HMC on higher-dimensional problems
- Implement NUTS (No-U-Turn Sampler) for automatic tuning
- Test on real-world Bayesian inference problems